# ClusterBP v2, этап 2: сравнение с mixture of experts

Вопрос рецензента о «personalized fusion»: даёт ли персонализация смешивания экспертов выигрыш над глобальным смешиванием?
В `clusterbp_v2.ipynb` покластерный блендинг (ClusterBP) не дал прироста. Здесь проверяется, помогает ли **обучаемый** гейт.

Эксперты — уже обученные в `clusterbp_v2.ipynb` модели каждого фолда: **GRU** (последние 20 измерений) и **XGB_full** (признаки АД + история событий).
Фолды, внутренняя валидация и фенотипы те же самые, тестовые пациенты не участвуют ни в каком обучении.

| Модель | Что это | Аналог в литературе |
|---|---|---|
| `Stack_global` | LR на logit-ах двух экспертов | обычный стекинг |
| `Gate_pheno` | LR на logit-ах экспертов и их произведениях с мягкой принадлежностью к фенотипам GDTW | обучаемая версия ClusterBP |
| `MoE_gate` | вес GRU = MLP(признаки пациента и фенотипа), p = w·p_GRU + (1−w)·p_XGB | гейт по «presentation data», Hurley et al., ACM HEALTH 2023 |
| `ResidualMoE` | базовая GRU на всех + эксперт XGBoost для каждого фенотипа, обученный на остатке (base_margin = logit GRU) | Lee & Hauskrecht, AIME 2022 |
| `Residual_global` | то же, но один эксперт на всех | контроль для `ResidualMoE` |

Гейты (`Stack_global`, `Gate_pheno`, `MoE_gate`) обучаются на внутренней валидации: на обучающей части предсказания экспертов in-sample и были бы переоптимистичны.
Эксперты остатка обучаются на обучающей части с ранней остановкой на валидации; базовая GRU на них in-sample — это ограничение схемы Lee & Hauskrecht.

In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, f1_score
from xgboost import XGBClassifier

from clusterbp_lib import (CACHE_DIR, RESULTS_DIR, SEED, N_FOLDS, N_CLUSTERS, FEATURE_SETS,
                           load_cached, fold_masks, add_phenotype_columns, best_threshold,
                           load_gru, load_xgb, gru_logits, sigmoid, logit, boot_eval, ci)

warnings.filterwarnings('ignore')
torch.set_num_threads(os.cpu_count())

MOE_TARGETS = ['A', 'B', 'OLD']
GATE_COLS = ['age', 'bmi', 'days_in_program', 'grp_experience', 'grp_control', 'n_meas', 'n_days',
             'sbp_mean', 'sbp_std', 'sbp_arv', 'sbp_slope', 'pd0', 'pd1', 'pd2']
N_BOOT = 1000

win, X_seq, seq_len, pheno_win, oof_v2 = load_cached()
print(f'Окон: {len(win):,}, пациентов: {win.pid.nunique():,}')

Окон: 420,355, пациентов: 9,413


In [2]:
def memberships(dist, temp=1.0):
    """Мягкая принадлежность к фенотипам: softmax(−расстояние GDTW)."""
    d = np.where(np.isnan(dist), np.nanmax(dist), dist)
    z = -(d - d.min(1, keepdims=True)) / temp
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_gate_lr(Z_va, y_va):
    sc = StandardScaler().fit(Z_va)
    lr = LogisticRegression(max_iter=3000, C=1.0).fit(sc.transform(Z_va), y_va)
    return lambda Z: lr.predict_proba(sc.transform(Z))[:, 1]


class Gate(nn.Module):
    def __init__(self, n_in, hidden=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, z, pg, px):
        w = torch.sigmoid(self.net(z).squeeze(-1))
        return w * pg + (1 - w) * px, w


def fit_moe_gate(Z, pg, px, y, seed, epochs=200, patience=10):
    """Гейт обучается на внутренней валидации; 20% её пациентов — для ранней остановки."""
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    med = np.nanmedian(Z, 0)
    Z = np.where(np.isnan(Z), med, Z)
    mu, sd = Z.mean(0), Z.std(0) + 1e-6
    Zs = ((Z - mu) / sd).astype(np.float32)
    idx = rng.permutation(len(y))
    n_es = len(y) // 5
    es, tr = idx[:n_es], idx[n_es:]
    T = lambda a: torch.from_numpy(np.asarray(a, dtype=np.float32))
    model = Gate(Z.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best, best_state, bad = np.inf, None, 0
    for _ in range(epochs):
        model.train()
        for b in np.array_split(rng.permutation(tr), max(1, len(tr) // 2048)):
            opt.zero_grad()
            p, _ = model(T(Zs[b]), T(pg[b]), T(px[b]))
            loss = nn.functional.binary_cross_entropy(p.clamp(1e-6, 1 - 1e-6), T(y[b]))
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            p, _ = model(T(Zs[es]), T(pg[es]), T(px[es]))
            l = nn.functional.binary_cross_entropy(p.clamp(1e-6, 1 - 1e-6), T(y[es])).item()
        if l < best - 1e-5:
            best, bad = l, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)

    def predict(Zn, pgn, pxn):
        Zn = (np.where(np.isnan(Zn), med, Zn) - mu) / sd
        with torch.no_grad():
            p, w = model(T(Zn), T(pgn), T(pxn))
        return p.numpy(), w.numpy()
    return predict


def fit_residual(X, base, y, tr, va, seed):
    """Эксперт XGBoost, обучаемый на остатке относительно базовой модели (base_margin = logit базы)."""
    m = XGBClassifier(n_estimators=1000, max_depth=3, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, tree_method='hist', eval_metric='aucpr',
                      early_stopping_rounds=50, n_jobs=-1, random_state=seed)
    m.fit(X[tr], y[tr], base_margin=base[tr], eval_set=[(X[va], y[va])],
          base_margin_eval_set=[base[va]], verbose=False)
    return m

## Обучение гейтов и экспертов остатка по фолдам

In [3]:
MODELS = ['Stack_global', 'Gate_pheno', 'MoE_gate', 'ResidualMoE', 'Residual_global']
oof = {t: {m: (np.full(len(win), np.nan), np.zeros(len(win), np.int8)) for m in MODELS} for t in MOE_TARGETS}
gate_weights, fold_rows = [], []
full_cols = FEATURE_SETS['full']
t_all = time.time()

for k in range(N_FOLDS):
    t_fold = time.time()
    tr, va, te = fold_masks(win, k)
    cl, dist = add_phenotype_columns(win, pheno_win, k)
    M = memberships(dist)
    X_full = win[full_cols].values
    Z_gate = win[GATE_COLS].values.astype(float)

    for t in MOE_TARGETS:
        y = win[f'y_{t}'].values
        gru, mu, sd, corr = load_gru(t, k)
        pg = sigmoid(gru_logits(gru, X_seq, seq_len, mu, sd) + corr)
        px = load_xgb('full', t, k).predict_proba(X_full)[:, 1]
        lg, lx = logit(pg), logit(px)
        res = {}

        S = np.column_stack([lg, lx])
        res['Stack_global'] = fit_gate_lr(S[va], y[va])(S)

        G = np.column_stack([lg, lx, lg[:, None] * M, lx[:, None] * M, M[:, :-1]])
        res['Gate_pheno'] = fit_gate_lr(G[va], y[va])(G)

        moe = fit_moe_gate(Z_gate[va], pg[va], px[va], y[va].astype(float), SEED + k)
        p_moe, w_moe = moe(Z_gate, pg, px)
        res['MoE_gate'] = p_moe
        for c in range(N_CLUSTERS):
            sel = te & (cl == c)
            gate_weights.append({'target': t, 'fold': k, 'cluster': c, 'w_gru_mean': float(w_moe[sel].mean()),
                                 'w_gru_sd': float(w_moe[sel].std())})

        base = lg
        g = fit_residual(X_full, base, y, tr, va, SEED + k)
        res['Residual_global'] = g.predict_proba(X_full, base_margin=base)[:, 1]
        p_res = np.empty(len(win))
        for c in np.unique(cl):
            in_c = cl == c
            if y[tr & in_c].sum() < 50 or y[va & in_c].sum() < 20:
                p_res[in_c] = res['Residual_global'][in_c]
                continue
            e = fit_residual(X_full, base, y, tr & in_c, va & in_c, SEED + k)
            p_res[in_c] = e.predict_proba(X_full[in_c], base_margin=base[in_c])[:, 1]
        res['ResidualMoE'] = p_res

        for m, score in res.items():
            thr, _ = best_threshold(y[va], score[va])
            pred = (score > thr).astype(np.int8)
            oof[t][m][0][te] = score[te]
            oof[t][m][1][te] = pred[te]
            fold_rows.append({'target': t, 'fold': k, 'model': m,
                              'auroc': roc_auc_score(y[te], score[te]),
                              'auprc': average_precision_score(y[te], score[te]),
                              'macro_f1': f1_score(y[te], pred[te], average='macro')})
    print(f'Фолд {k}: {time.time() - t_fold:.0f} с')

print(f'Всего: {(time.time() - t_all) / 60:.1f} мин')
moe_df = win[['pid', 'cut', 'fold']].copy()
for t in MOE_TARGETS:
    for m, (score, pred) in oof[t].items():
        moe_df[f'{t}__{m}__score'] = score
        moe_df[f'{t}__{m}__pred'] = pred
moe_df.to_parquet(os.path.join(CACHE_DIR, 'oof_moe.parquet'))
pd.DataFrame(fold_rows).to_csv(os.path.join(RESULTS_DIR, 'moe_fold_metrics.csv'), index=False)
pd.DataFrame(gate_weights).to_csv(os.path.join(RESULTS_DIR, 'moe_gate_weights.csv'), index=False)

Фолд 0: 34 с


Фолд 1: 45 с


Фолд 2: 45 с


Фолд 3: 56 с


Фолд 4: 49 с
Всего: 3.8 мин


## Итог: персонализированные схемы против глобальных

Для сравнения добавлены модели из `clusterbp_v2.ipynb` на тех же окнах: `XGB_full`, `GRU`, `Blend_global_full`, `ClusterBP_full`.
Разности — бутстреп по пациентам, 95% ДИ.

In [4]:
REF = ['XGB_full', 'GRU', 'Blend_global_full', 'ClusterBP_full']
COMPARE = [('Gate_pheno', 'Stack_global'), ('MoE_gate', 'Stack_global'), ('ResidualMoE', 'Residual_global'),
           ('MoE_gate', 'Blend_global_full'), ('ResidualMoE', 'XGB_full'), ('Stack_global', 'XGB_full')]
fold_df = pd.DataFrame(fold_rows)
tables, comps = {}, []
for t in MOE_TARGETS:
    y = win[f'y_{t}'].values.astype(float)
    names = REF + MODELS
    models = [(oof_v2[f'{t}__{m}__score'].values, oof_v2[f'{t}__{m}__pred'].values.astype(float)) for m in REF]
    models += [(oof[t][m][0], oof[t][m][1].astype(float)) for m in MODELS]
    B = boot_eval(y, win['pid'].values, models, n_boot=N_BOOT)
    np.save(os.path.join(CACHE_DIR, f'bootstrap_moe_{t}.npy'), B)
    rows = []
    for j, m in enumerate(names):
        s, p = models[j]
        rows.append({'model': m, 'AUROC': roc_auc_score(y, s), 'AUROC 95% ДИ': ci(B[:, j, 0]),
                     'AUPRC': average_precision_score(y, s), 'AUPRC 95% ДИ': ci(B[:, j, 1]),
                     'Brier': brier_score_loss(y, s),
                     'macro_F1': f1_score(y, p, average='macro'), 'macro_F1 95% ДИ': ci(B[:, j, 2])})
    tables[t] = pd.DataFrame(rows).set_index('model').sort_values('AUPRC', ascending=False)
    tables[t].to_csv(os.path.join(RESULTS_DIR, f'moe_metrics_{t}.csv'))
    for a, b in COMPARE:
        ia, ib = names.index(a), names.index(b)
        for mi, metric in enumerate(['AUROC', 'AUPRC', 'macro_F1']):
            d = B[:, ia, mi] - B[:, ib, mi]
            comps.append({'target': t, 'comparison': f'{a} − {b}', 'metric': metric,
                          'delta': tables[t].loc[a, metric] - tables[t].loc[b, metric], '95% ДИ': ci(d),
                          'p (двусторонний)': min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))})
comp_df = pd.DataFrame(comps)
comp_df.to_csv(os.path.join(RESULTS_DIR, 'moe_comparisons.csv'), index=False)

pd.set_option('display.width', 250)
for t, tab in tables.items():
    print(f'\n=== {t} ===')
    display(tab.round(3))


=== A ===


,AUROC,AUROC 95% ДИ,AUPRC,AUPRC 95% ДИ,Brier,macro_F1,macro_F1 95% ДИ
model,,,,,,,
Gate_pheno,0.897,[0.891; 0.903],0.452,[0.422; 0.478],0.033,0.716,[0.705; 0.726]
Stack_global,0.897,[0.891; 0.903],0.451,[0.422; 0.478],0.033,0.716,[0.705; 0.726]
MoE_gate,0.897,[0.890; 0.902],0.451,[0.422; 0.477],0.033,0.715,[0.705; 0.725]
Blend_global_full,0.896,[0.890; 0.902],0.449,[0.420; 0.476],0.033,0.715,[0.704; 0.725]
ClusterBP_full,0.896,[0.890; 0.902],0.449,[0.420; 0.476],0.033,0.714,[0.704; 0.724]
XGB_full,0.896,[0.890; 0.902],0.448,[0.419; 0.474],0.033,0.714,[0.704; 0.723]
Residual_global,0.892,[0.885; 0.897],0.438,[0.408; 0.464],0.033,0.710,[0.700; 0.719]
ResidualMoE,0.890,[0.884; 0.896],0.435,[0.406; 0.462],0.034,0.709,[0.698; 0.718]
GRU,0.872,[0.865; 0.879],0.403,[0.374; 0.430],0.034,0.701,[0.690; 0.711]



=== B ===


,AUROC,AUROC 95% ДИ,AUPRC,AUPRC 95% ДИ,Brier,macro_F1,macro_F1 95% ДИ
model,,,,,,,
Gate_pheno,0.911,[0.907; 0.915],0.679,[0.662; 0.694],0.067,0.785,[0.778; 0.792]
Stack_global,0.911,[0.907; 0.914],0.678,[0.661; 0.694],0.067,0.785,[0.778; 0.792]
MoE_gate,0.911,[0.907; 0.914],0.677,[0.661; 0.693],0.067,0.785,[0.778; 0.791]
Blend_global_full,0.910,[0.906; 0.914],0.677,[0.660; 0.693],0.067,0.784,[0.777; 0.791]
ClusterBP_full,0.910,[0.906; 0.914],0.677,[0.660; 0.692],0.067,0.784,[0.777; 0.791]
XGB_full,0.910,[0.906; 0.914],0.676,[0.658; 0.691],0.067,0.784,[0.777; 0.791]
Residual_global,0.908,[0.904; 0.912],0.671,[0.654; 0.688],0.068,0.782,[0.776; 0.789]
ResidualMoE,0.907,[0.903; 0.911],0.669,[0.652; 0.685],0.068,0.782,[0.775; 0.788]
GRU,0.895,[0.890; 0.899],0.643,[0.626; 0.660],0.071,0.773,[0.766; 0.780]



=== OLD ===


,AUROC,AUROC 95% ДИ,AUPRC,AUPRC 95% ДИ,Brier,macro_F1,macro_F1 95% ДИ
model,,,,,,,
Gate_pheno,0.928,[0.924; 0.933],0.526,[0.500; 0.550],0.026,0.743,[0.734; 0.753]
Stack_global,0.928,[0.924; 0.933],0.526,[0.500; 0.550],0.026,0.744,[0.734; 0.753]
ClusterBP_full,0.928,[0.924; 0.933],0.526,[0.499; 0.550],0.026,0.743,[0.734; 0.753]
MoE_gate,0.929,[0.924; 0.933],0.526,[0.499; 0.550],0.026,0.744,[0.734; 0.754]
Blend_global_full,0.928,[0.924; 0.933],0.525,[0.498; 0.549],0.026,0.743,[0.734; 0.753]
XGB_full,0.928,[0.923; 0.932],0.524,[0.497; 0.548],0.026,0.742,[0.732; 0.752]
Residual_global,0.925,[0.920; 0.929],0.512,[0.485; 0.536],0.027,0.739,[0.729; 0.748]
ResidualMoE,0.924,[0.919; 0.928],0.506,[0.478; 0.530],0.027,0.736,[0.727; 0.746]
GRU,0.913,[0.908; 0.918],0.470,[0.442; 0.495],0.028,0.727,[0.716; 0.736]


In [5]:
display(comp_df[comp_df.metric != 'AUROC'].round(4))
gw = pd.DataFrame(gate_weights)
display(gw.groupby(['target', 'cluster'])[['w_gru_mean', 'w_gru_sd']].mean().round(3))

,target,comparison,metric,delta,95% ДИ,p (двусторонний)
1,A,Gate_pheno − Stack_global,AUPRC,0.0003,[-0.000; 0.001],0.406
2,A,Gate_pheno − Stack_global,macro_F1,0.0005,[-0.000; 0.002],0.376
4,A,MoE_gate − Stack_global,AUPRC,-0.0007,[-0.002; 0.000],0.182
5,A,MoE_gate − Stack_global,macro_F1,-0.0003,[-0.002; 0.001],0.630
7,A,ResidualMoE − Residual_global,AUPRC,-0.0027,[-0.004; -0.001],0.000
8,A,ResidualMoE − Residual_global,macro_F1,-0.0014,[-0.003; -0.000],0.034
10,A,MoE_gate − Blend_global_full,AUPRC,0.0013,[-0.000; 0.003],0.068
11,A,MoE_gate − Blend_global_full,macro_F1,0.0003,[-0.001; 0.001],0.588
13,A,ResidualMoE − XGB_full,AUPRC,-0.0127,[-0.017; -0.008],0.000
14,A,ResidualMoE − XGB_full,macro_F1,-0.0055,[-0.008; -0.003],0.000


w_gru_mean  w_gru_sd
target cluster                      
A      0             0.292     0.140
       1             0.270     0.131
       2             0.270     0.133
B      0             0.189     0.100
       1             0.183     0.105
       2             0.176     0.098
OLD    0             0.274     0.096
       1             0.265     0.099
       2             0.255     0.098